# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  evenements_volumineux.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [2]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (17022, 23)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [3]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5167,18853464.16,3648.82,1221.80,26591
Audio,3960,9722062.63,2455.07,1108.27,15056
Mobilier,3942,9030796.68,2290.92,782.73,18255
Accessoire,3878,8338197.08,2150.13,886.12,14960


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [4]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie  montant  ca_categorie  part_dans_categorie
0  Informatique   481.98   18853464.16               0.0026
1      Mobilier   140.96    9030796.68               0.0016
2         Audio  2073.36    9722062.63               0.0213
3  Informatique  2281.00   18853464.16               0.0121
4      Mobilier  2116.10    9030796.68               0.0234

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [5]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,5746,P0027,0.165
telephone,2871,P0015,0.166
web,8405,P0038,0.171


### Partie autonome

In [6]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.

revenue_table = sales.pivot_table(index='categorie', columns='canal', values='montant',
                                  aggfunc='sum', margins=True, margins_name='Total',
                                  fill_value=0).round(0)
print(revenue_table)

# Contrôle : pivot_table écarte en silence les lignes dont la catégorie ou le canal est manquant
# n_excluded = int(sales[['categorie', 'canal']].isna().any(axis=1).sum())
# expected_total = sales.dropna(subset=['categorie', 'canal'])['montant'].sum()
# print(f'\nLignes exclues du tableau (catégorie ou canal manquant) : {n_excluded}')
# assert np.isclose(revenue_table.loc['Total', 'Total'], expected_total, atol=1), "le total en marge est faux"

canal           boutique   telephone         web       Total
categorie                                                   
Accessoire     2356697.0   2622340.0   3359160.0   8338197.0
Audio          2947854.0    915608.0   5858601.0   9722063.0
Informatique   6868597.0   5093331.0   6891536.0  18853464.0
Mobilier       3579864.0   2740733.0   2710199.0   9030797.0
Total         15753012.0  11372013.0  18819496.0  45944521.0


In [7]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.

# montant_livre : seul le chiffre d'affaires réellement livré (statut 'livre')
sales_for_clients = sales.assign(montant_livre=sales['montant'].where(sales['statut'] == 'livre', 0))

customer_summary = sales_for_clients.groupby('id_client').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique'),
    ca_livre=('montant_livre', 'sum'),
).round(2)

top_10 = customer_summary.nlargest(10, 'ca_total')
print(top_10)

# groupby écarte en silence les lignes sans identifiant client (dropna=True par défaut)
print('\nLignes sans id_client (absentes du tableau) :', int(sales['id_client'].isna().sum()))

# Le classement change-t-il si l'on ne compte que les commandes livrées ?
top_10_delivered = customer_summary.nlargest(10, 'ca_livre')
common = len(set(top_10.index) & set(top_10_delivered.index))
print(f'Clients communs aux deux classements (tout / livré) : {common} sur 10')

           nb_commandes    ca_total  panier_moyen   premiere_commande   derniere_commande  nb_categories    ca_livre
id_client                                                                                                           
C00696                7  1110754.11     158679.16 2023-03-13 00:00:00 2024-11-19 00:00:00              3  1110574.06
C00279               10  1094384.59     109438.46 2023-01-24 00:00:00 2024-11-05 00:00:00              4     8806.73
C02327                4   955731.99     238933.00 2023-10-31 00:00:00 2024-12-28 14:36:00              2      434.52
C02631               10   926320.69      92632.07 2023-01-12 00:00:00 2024-12-19 00:00:00              4   924445.79
C02285                7   857687.95     122526.85 2023-01-10 10:43:00 2024-12-27 00:00:00              2     6048.38
C00355               13   820029.75      63079.21 2023-04-09 16:29:00 2024-09-25 00:00:00              4     8340.20
C00806                5   785897.24     157179.45 2023-05-03 00:

In [8]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?

median_by_category = sales.groupby('categorie')['montant'].transform('median')
sales['commande_haute'] = sales['montant'] > 1.5 * median_by_category

share = sales['commande_haute'].mean()
print(f'Part des commandes au-dessus de 150 % de la médiane de leur catégorie : {share:.1%}')
print('\nPar catégorie :')
print(sales.groupby('categorie')['commande_haute'].mean().round(3))

# Une catégorie manquante n'a pas de médiane : la comparaison renvoie False pour ces lignes
print('\nLignes sans catégorie (jamais signalées) :', int(sales['categorie'].isna().sum()))

Part des commandes au-dessus de 150 % de la médiane de leur catégorie : 33.5%

Par catégorie :
categorie
Accessoire      0.281
Audio           0.362
Informatique    0.357
Mobilier        0.341
Name: commande_haute, dtype: float64

Lignes sans catégorie (jamais signalées) : 75


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [9]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 17022
Après jointure produits : 17022
Lignes sans produit correspondant : 75 (0.44%)
Identifiants fautifs : ['P9999']


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [10]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
17022 lignes -> 17318 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [11]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).

# Point de départ : le jeu de la séance 4 contient déjà des colonnes issues du référentiel
# produits. On les retire pour les rejoindre proprement et valider la jointure.
base_sales = load_dataset('ventes_transformees')
base_sales['date_commande'] = pd.to_datetime(base_sales['date_commande'], errors='coerce')
already_joined = [c for c in products.columns if c != 'id_produit' and c in base_sales.columns]
base_sales = base_sales.drop(columns=already_joined)

analytic = base_sales
steps = [{'etape': 'ventes (départ)', 'lignes': len(analytic), 'cles_orphelines': 0, 'doublons_retires': 0}]

for label, dimension, key, suffix in [('produits', products, 'id_produit', '_produit'),
                                      ('clients', customers, 'id_client', '_client'),
                                      ('magasins', stores, 'id_magasin', '_magasin')]:
    # 1. clé primaire unique à droite : on retire les doublons et on les compte
    dimension_unique = dimension.drop_duplicates(key)
    n_duplicates = len(dimension) - len(dimension_unique)

    # 2. clés orphelines : présentes (ou manquantes) à gauche, absentes à droite
    is_orphan = ~analytic[key].isin(dimension_unique[key])

    # 3. jointure validée : une erreur est levée si la clé de droite n'est pas unique
    columns_before = set(analytic.columns)
    n_before = len(analytic)
    analytic = analytic.merge(dimension_unique, on=key, how='left',
                              validate='many_to_one', suffixes=('', suffix))
    assert len(analytic) == n_before, f'la jointure {label} a changé le nombre de lignes'

    # 4. choix documenté : on CONSERVE les lignes orphelines (aucune perte de chiffre d'affaires),
    #    on étiquette les attributs texte 'Inconnu' et on ajoute un indicateur.
    new_columns = [c for c in analytic.columns if c not in columns_before]
    for column in new_columns:
        if analytic[column].dtype == object or str(analytic[column].dtype) == 'string':
            analytic[column] = analytic[column].fillna('Inconnu')
    analytic[f'orphelin_{label}'] = is_orphan.to_numpy()

    steps.append({'etape': f'+ {label}', 'lignes': len(analytic),
                  'cles_orphelines': int(is_orphan.sum()), 'doublons_retires': n_duplicates})

print(pd.DataFrame(steps).to_string(index=False))
print('\nDimensions du DataFrame analytique :', analytic.shape)

          etape  lignes  cles_orphelines  doublons_retires
ventes (départ)   17022                0                 0
     + produits   17022               75                 0
      + clients   17022                0                 0
     + magasins   17022                0                 0

Dimensions du DataFrame analytique : (17022, 34)


In [12]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.

def safe_merge(left, right, **kwargs):
    """Jointure qui échoue bruyamment si un 'left join' change le nombre de lignes.

    Accepte les mêmes arguments que DataFrame.merge (on, how, validate, suffixes...).
    Affiche le taux de lignes de gauche sans correspondance (jointures left et outer)
    et le range dans `resultat.attrs['unmatched_rate']`.
    """
    if kwargs.get('indicator'):
        raise ValueError("safe_merge gère déjà l'indicateur en interne")
    how = kwargs.get('how', 'inner')
    n_before = len(left)

    merged = left.merge(right, indicator='_merge_check', **kwargs)
    n_after = len(merged)

    if how == 'left' and n_after != n_before:
        raise ValueError(f"left join : {n_before} lignes -> {n_after}. "
                         "La clé de droite est-elle unique ? (essayer validate='many_to_one')")

    unmatched_rate = None
    if how in ('left', 'outer'):
        unmatched_rate = float((merged['_merge_check'] == 'left_only').sum() / max(n_before, 1))
        print(f'safe_merge : {n_before} -> {n_after} lignes, '
              f'clés non appariées : {unmatched_rate:.2%}')
    else:
        print(f'safe_merge ({how}) : {n_before} -> {n_after} lignes')

    merged = merged.drop(columns='_merge_check')
    merged.attrs['unmatched_rate'] = unmatched_rate
    return merged


# Démonstration 1 : jointure saine, le taux de clés orphelines est signalé
checked = safe_merge(base_sales, products.drop_duplicates('id_produit'), on='id_produit',
                     how='left', validate='many_to_one', suffixes=('', '_produit'))

# Démonstration 2 : table de droite dégradée, l'explosion de lignes est interceptée
try:
    safe_merge(base_sales, customers_with_duplicates, on='id_client', how='left')
except ValueError as error:
    print('ValueError attendue :', error)

safe_merge : 17022 -> 17022 lignes, clés non appariées : 0.44%
ValueError attendue : left join : 17022 lignes -> 17318. La clé de droite est-elle unique ? (essayer validate='many_to_one')


---
## Atelier 6.3 — Formats long et large (35 min)

In [13]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{month:02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,395173.0,316303.0,356174.0,1321830.0,562044.0,352159.0,356143.0,1455170.0,520916.0,1202412.0,1001896.0,497977.0
Audio,1202283.0,687507.0,419522.0,1230168.0,452758.0,1191852.0,603419.0,1077320.0,445789.0,457582.0,1213118.0,740745.0
Informatique,2012495.0,1678876.0,3071357.0,724833.0,2464321.0,791228.0,3116701.0,1158080.0,602341.0,635594.0,824967.0,1772672.0
Mobilier,964488.0,346240.0,399305.0,1209281.0,1424455.0,965760.0,386091.0,209440.0,1538760.0,320933.0,762646.0,503397.0


In [14]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,395173.0
1,Audio,mois_01,1202283.0
2,Informatique,mois_01,2012495.0
3,Mobilier,mois_01,964488.0
4,Accessoire,mois_02,316303.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [15]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.

# pivot (et non pivot_table) : il ne sait pas agréger, il suppose une valeur par couple
wide_back = long.pivot(index='categorie', columns='mois', values='ca')
wide_back.columns.name = None          # pivot nomme l'axe des colonnes : on retire ce nom

pd.testing.assert_frame_equal(wide_back, wide)
print('OK : le tableau reconstruit est identique au tableau de départ')

# Équivalent : long.set_index(['categorie', 'mois'])['ca'].unstack()

OK : le tableau reconstruit est identique au tableau de départ


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [16]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [17]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.

UNKNOWN_KEY = 'INCONNU'


def add_unknown_members(fact_table, dimension, key, label=UNKNOWN_KEY):
    """Garantit l'intégrité référentielle sans perdre de lignes de faits.

    Les clés manquantes de la table de faits deviennent `label`. Chaque clé absente de la
    dimension reçoit une ligne « Inconnu » (attributs texte = 'Inconnu', numériques vides).
    Renvoie (table_de_faits, dimension, journal).
    """
    fact_table = fact_table.copy()
    n_missing = int(fact_table[key].isna().sum())
    fact_table[key] = fact_table[key].fillna(label)

    orphan_keys = pd.Index(fact_table[key].unique()).difference(pd.Index(dimension[key]))
    placeholders = pd.DataFrame({key: orphan_keys})
    for column in dimension.columns.drop(key):
        is_text = (pd.api.types.is_object_dtype(dimension[column])
                   or pd.api.types.is_string_dtype(dimension[column]))
        placeholders[column] = 'Inconnu' if is_text else np.nan
    dimension = pd.concat([dimension, placeholders], ignore_index=True)

    log = {'cles_manquantes_remplacees': n_missing,
           'cles_orphelines_ajoutees': int(len(orphan_keys)),
           'lignes_de_faits_concernees': int(fact_table[key].isin(orphan_keys).sum())}
    return fact_table, dimension, log


STAR_DIR = PROCESSED_DIR / 'star'
STAR_DIR.mkdir(parents=True, exist_ok=True)

assert base_sales['date_commande'].notna().all(), 'dates manquantes : à traiter avant le modèle en étoile'

# --- Table de faits : clés étrangères + mesures, rien d'autre
fact_sales = base_sales[['id_client', 'id_produit', 'id_magasin', 'quantite',
                         'prix_unitaire', 'remise_pct', 'montant', 'statut']].copy()
fact_sales['cle_date'] = base_sales['date_commande'].dt.strftime('%Y%m%d').astype(int)
fact_sales['montant_net'] = fact_sales['montant'] * (1 - fact_sales['remise_pct'].fillna(0) / 100)
fact_sales = fact_sales[['cle_date', 'id_client', 'id_produit', 'id_magasin', 'quantite',
                         'prix_unitaire', 'remise_pct', 'montant', 'montant_net', 'statut']]

# --- Dimensions : dédoublonnées sur leur clé primaire (on distingue doublons exacts et conflits)
star_log = {'lignes_faits_depart': len(fact_sales)}
dimensions_raw = {'id_client': ('dim_customer', customers),
                  'id_produit': ('dim_product', products),
                  'id_magasin': ('dim_store', stores)}
dimensions = {}
for key, (name, source) in dimensions_raw.items():
    unique_rows = source.drop_duplicates()
    conflicts = len(unique_rows) - len(unique_rows.drop_duplicates(key))
    dimension = source.drop_duplicates(key).reset_index(drop=True)
    print(f'{name:<13} {len(source):>6} -> {len(dimension):>6} lignes '
          f'({len(source) - len(unique_rows)} doublons exacts, {conflicts} conflits à arbitrer)')
    fact_sales, dimension, log = add_unknown_members(fact_sales, dimension, key)
    star_log[name] = log
    dimensions[key] = dimension
dimensions['cle_date'] = dim_date

dim_customer, dim_product, dim_store = (dimensions['id_client'], dimensions['id_produit'],
                                        dimensions['id_magasin'])

# --- Contrôles : aucune ligne ni aucun euro perdu, intégrité référentielle
assert len(fact_sales) == len(base_sales)
assert np.isclose(fact_sales['montant'].sum(), base_sales['montant'].sum())
for key, dimension in dimensions.items():
    n_orphans = int((~fact_sales[key].isin(dimension[key])).sum())
    print(f'clés orphelines {key:<11} : {n_orphans}')
    assert n_orphans == 0

# --- Export Parquet (repli CSV si pyarrow est absent)
def save_star_table(df, name):
    try:
        df.to_parquet(STAR_DIR / f'{name}.parquet', index=False)
    except ImportError:
        df.to_csv(STAR_DIR / f'{name}.csv', index=False)
    print('écrit :', name, df.shape)

for name, table in [('fact_sales', fact_sales), ('dim_date', dim_date), ('dim_customer', dim_customer),
                    ('dim_product', dim_product), ('dim_store', dim_store)]:
    save_star_table(table, name)
print('\nJournal :', star_log)

dim_customer    3060 ->   3060 lignes (0 doublons exacts, 0 conflits à arbitrer)
dim_product       39 ->     39 lignes (0 doublons exacts, 0 conflits à arbitrer)
dim_store          9 ->      9 lignes (0 doublons exacts, 0 conflits à arbitrer)
clés orphelines id_client   : 0
clés orphelines id_produit  : 0
clés orphelines id_magasin  : 0
clés orphelines cle_date    : 0
écrit : fact_sales (17022, 10)
écrit : dim_date (730, 10)
écrit : dim_customer (3060, 5)
écrit : dim_product (40, 5)
écrit : dim_store (9, 5)

Journal : {'lignes_faits_depart': 17022, 'dim_customer': {'cles_manquantes_remplacees': 0, 'cles_orphelines_ajoutees': 0, 'lignes_de_faits_concernees': 0}, 'dim_product': {'cles_manquantes_remplacees': 0, 'cles_orphelines_ajoutees': 1, 'lignes_de_faits_concernees': 75}, 'dim_store': {'cles_manquantes_remplacees': 0, 'cles_orphelines_ajoutees': 0, 'lignes_de_faits_concernees': 0}}


In [18]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

def check_referential_integrity(fact_table, dimensions):
    """dimensions : dict {nom_colonne_cle: dataframe_dimension}.

    Pour chaque couple, compte les clés orphelines, les clés manquantes dans les faits
    et les doublons de clé dans la dimension. Lève une ValueError si un compte est non nul.
    """
    results, problems = {}, []
    for key, dimension in dimensions.items():
        for name, frame in [('faits', fact_table), ('dimension', dimension)]:
            if key not in frame.columns:
                raise KeyError(f"colonne '{key}' absente de la table de {name}")
        n_missing = int(fact_table[key].isna().sum())
        n_orphans = int((fact_table[key].notna() & ~fact_table[key].isin(dimension[key])).sum())
        n_duplicates = int(dimension[key].duplicated().sum())
        results[key] = {'orphelines': n_orphans, 'manquantes': n_missing,
                        'doublons_dimension': n_duplicates}
        print(f'{key:<12} orphelines={n_orphans:<6} manquantes={n_missing:<6} '
              f'doublons_dimension={n_duplicates}')
        if n_orphans or n_missing or n_duplicates:
            problems.append(f'{key} ({n_orphans} orphelines, {n_missing} manquantes, '
                            f'{n_duplicates} doublons)')
    if problems:
        raise ValueError('Intégrité référentielle violée : ' + ' ; '.join(problems))
    return results


# Sur le modèle livré : tout doit passer
check_referential_integrity(fact_sales, dimensions)

# Sur un modèle volontairement cassé (on retire un client de la dimension) : l'erreur doit se lever
broken = {**dimensions, 'id_client': dim_customer.iloc[1:]}
try:
    check_referential_integrity(fact_sales, broken)
except ValueError as error:
    print('\nValueError attendue :', error)

id_client    orphelines=0      manquantes=0      doublons_dimension=0
id_produit   orphelines=0      manquantes=0      doublons_dimension=0
id_magasin   orphelines=0      manquantes=0      doublons_dimension=0
cle_date     orphelines=0      manquantes=0      doublons_dimension=0
id_client    orphelines=6      manquantes=0      doublons_dimension=0
id_produit   orphelines=0      manquantes=0      doublons_dimension=0
id_magasin   orphelines=0      manquantes=0      doublons_dimension=0
cle_date     orphelines=0      manquantes=0      doublons_dimension=0

ValueError attendue : Intégrité référentielle violée : id_client (6 orphelines, 0 manquantes, 0 doublons)


---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.